# Determinants and Inverses

| Difficulty | ███░░░░░░░ 3/10 |
| :--- | :--- |
| Prerequisites | 01.04 |
| Time | ~55 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/05_determinants_and_inverses.ipynb)

---

## 🎯 Learning Objective

Understand what the determinant measures geometrically, when a matrix can be inverted, and how the Moore-Penrose pseudoinverse generalizes inversion to non-square matrices.

---

## 📐 Theory

### The determinant

For a $2\times2$ matrix, $\det\begin{pmatrix}a&b\\c&d\end{pmatrix} = ad-bc$. Geometrically,
$|\det(A)|$ is the factor by which $A$ scales area (in 2D) or volume (in higher dimensions);
the *sign* tells you whether orientation is preserved (positive) or flipped, like a reflection
(negative). Key properties:

$$\det(AB) = \det(A)\det(B) \qquad \det(A^\top) = \det(A) \qquad \det(A) = 0 \iff A \text{ is singular}$$

A matrix is **singular** (non-invertible) exactly when its determinant is zero — geometrically,
it collapses space into a lower dimension, and no transformation can undo that information loss.

### The inverse

$A^{-1}$ is the matrix satisfying $AA^{-1} = A^{-1}A = I$. For $2\times2$:

$$A^{-1} = \frac{1}{\det(A)}\begin{pmatrix}d & -b \\ -c & a\end{pmatrix}$$

Notice the formula divides by $\det(A)$ — this is *why* a zero determinant means no inverse
exists, and why a determinant *close* to zero causes numerical trouble (dividing by a tiny
number amplifies errors).

### The Moore-Penrose pseudoinverse

Non-square or singular matrices don't have a true inverse, but the **pseudoinverse** $A^+$
generalizes the idea: it gives the least-squares "best" solution to $A\mathbf{x}=\mathbf{b}$ for
overdetermined systems, and the minimum-norm solution for underdetermined ones. It's computed
via the SVD (which we'll properly meet in [Notebook 01.07](07_SVD_and_PCA.ipynb)):
$A^+ = V\Sigma^+U^\top$.

---

## 👁️ Visual Intuition

The determinant is literally the area-scaling factor of a transformation: watch the unit
square's area change (and its orientation flip, for a reflection) as we apply different matrices.

In [ ]:
# |det(A)| = area scaling factor; sign of det(A) = orientation preserved (+) or flipped (-)
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

unit_square = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])  # closed polygon for plotting

matrices = {
    "Stretch (det=3)": np.array([[3.0, 0.0], [0.0, 1.0]]),
    "Shear (det=1, area preserved)": np.array([[1.0, 0.8], [0.0, 1.0]]),
    "Reflection (det=-1)": np.array([[0.0, 1.0], [1.0, 0.0]]),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, (title, A) in zip(axes, matrices.items()):
    transformed = A @ unit_square
    ax.plot(*unit_square, color="gray", ls="--", label="unit square")
    ax.fill(*transformed, alpha=0.4, color="#4C72B0")
    ax.plot(*transformed, color="#4C72B0", label="transformed")
    ax.set_title(f"{title}\ndet={np.linalg.det(A):.2f}")
    ax.set_aspect('equal'); ax.set_xlim(-1.5, 3.5); ax.set_ylim(-1.5, 3.5)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

Cofactor expansion for $2\times2$/$3\times3$ determinants, the $2\times2$ inverse formula, and
the pseudoinverse for a non-square matrix — each checked against NumPy.

In [ ]:
# Determinant by cofactor expansion, 2x2 inverse formula, and the pseudoinverse
import numpy as np

def det_2x2(A):
    return A[0, 0] * A[1, 1] - A[0, 1] * A[1, 0]

def det_3x3(A):
    # Cofactor expansion along the first row
    return (A[0, 0] * det_2x2(A[1:, [1, 2]])
            - A[0, 1] * det_2x2(A[1:, [0, 2]])
            + A[0, 2] * det_2x2(A[1:, [0, 1]]))

def inverse_2x2(A):
    d = det_2x2(A)
    if abs(d) < 1e-12:
        raise ValueError("matrix is singular (det ~ 0), cannot invert")
    return (1.0 / d) * np.array([[A[1, 1], -A[0, 1]], [-A[1, 0], A[0, 0]]])

A2 = np.array([[4.0, 7.0], [2.0, 6.0]])
A3 = np.array([[6.0, 1.0, 1.0], [4.0, -2.0, 5.0], [2.0, 8.0, 7.0]])

print("det_2x2 vs np.linalg.det:", det_2x2(A2), "vs", round(float(np.linalg.det(A2)), 6))
print("det_3x3 vs np.linalg.det:", det_3x3(A3), "vs", round(float(np.linalg.det(A3)), 6))
print("inverse_2x2 matches np.linalg.inv:", np.allclose(inverse_2x2(A2), np.linalg.inv(A2)))

print("\n--- Pseudoinverse for a non-square (overdetermined) matrix ---")
A_rect = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])   # 3x2, no true inverse exists
A_pinv = np.linalg.pinv(A_rect)
print("A_rect shape:", A_rect.shape, " A_pinv shape:", A_pinv.shape)
print("A_pinv @ A_rect (should be close to I_2):\n", np.round(A_pinv @ A_rect, 6))

## 🤖 Why This Matters for AI

**Near-singular matrices cause numerical instability.** When $\det(A)$ is close to zero, the
formula for $A^{-1}$ divides by a tiny number, amplifying any noise in the input into a huge
error in the output. This is directly relevant to **normalizing flows**, a family of generative
models that must use *invertible* transformations by construction — flow architectures are
carefully designed (e.g. using triangular Jacobians) specifically so their determinants stay
well away from zero and cheap to compute.

In [ ]:
# Near-singular matrices amplify tiny input errors into huge output errors
import numpy as np

def relative_error(x_true, x_approx):
    return np.linalg.norm(x_true - x_approx) / np.linalg.norm(x_true)

# A matrix that is ALMOST singular: its two rows are nearly parallel
A_nearly_singular = np.array([[1.0, 1.0], [1.0, 1.0001]])
print("det(A) =", np.linalg.det(A_nearly_singular), " (very close to zero)")

b = np.array([2.0, 2.0001])
x_true = np.linalg.solve(A_nearly_singular, b)

b_perturbed = b + np.array([1e-4, 0.0])       # a TINY perturbation to b
x_perturbed = np.linalg.solve(A_nearly_singular, b_perturbed)

print("\nOriginal solution:  ", np.round(x_true, 4))
print("Perturbed solution: ", np.round(x_perturbed, 4))
print(f"Input perturbation size:  {np.linalg.norm(b_perturbed - b):.6f}")
print(f"Output change size:       {np.linalg.norm(x_perturbed - x_true):.6f}")
print("\nA tiny nudge to b caused a huge swing in x -- this sensitivity is exactly what the")
print("CONDITION NUMBER (Notebook 01.06) will let us quantify precisely.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Compute $\det\begin{pmatrix}3&8\\4&6\end{pmatrix}$ and its inverse by hand, then verify with
   `det_2x2`/`inverse_2x2`. As a sanity check, confirm $AA^{-1}=I$ exactly (up to floating-point
   rounding) — the defining property any correctly-computed inverse must satisfy.

<details>
<summary>💡 Solution</summary>

$\det(A)=3\cdot6-8\cdot4=18-32=-14$. Using the $2\times2$ inverse formula,
$$A^{-1}=\frac{1}{-14}\begin{pmatrix}6&-8\\-4&3\end{pmatrix}=
\begin{pmatrix}-0.4286&0.5714\\0.2857&-0.2143\end{pmatrix},$$
matching `inverse_2x2(A)` and `np.linalg.inv(A)` exactly. Check: $AA^{-1}=I$ to floating-point
precision — confirmed directly, not assumed.

</details>

### 💭 UNDERSTAND
2. $\det(AB)=\det(A)\det(B)$ is a true, exact identity for any square $A,B$ of the same size.
   A common misconception treats the determinant as *linear*, expecting
   $\det(A+B)=\det(A)+\det(B)$ too. Without computing anything yet, do you expect this additive
   version to hold? Test both identities on two random $3\times3$ matrices, then identify the
   one respect in which the determinant *is* linear (hint: consider $\det(cA)$ for a scalar
   $c$).

<details>
<summary>💡 Solution</summary>

The multiplicative identity holds exactly (verified: $\det(AB)$ and $\det(A)\det(B)$ agree to
floating-point precision on random $3\times3$ matrices), but the additive one fails — testing
several random pairs gives $\det(A+B)$ values with no consistent relationship to
$\det(A)+\det(B)$ at all (in one trial, $0.49$ versus $1.15$; in another, $-4.85$ versus $0.69$
— not even the same sign). The determinant is **not** linear in the "add two matrices" sense,
but it *is* homogeneous of degree $n$ in a single matrix's scale: $\det(cA)=c^n\det(A)$ for an
$n\times n$ matrix (verified: scaling a $3\times3$ matrix by $c=2.5$ scales its determinant by
exactly $2.5^3=15.625$, matching numerically). This makes geometric sense: scaling *every*
dimension of an $n$-dimensional object by $c$ multiplies its volume by $c^n$, not $c$ — and the
determinant is fundamentally a volume-scaling factor, not a sum of anything.

</details>

### ✏️ DERIVE
3. Derive the general inverse formula $A^{-1} = \frac{1}{\det(A)}\,\text{adj}(A)$ (where
   $\text{adj}(A)$, the *adjugate*, is the matrix of cofactors, transposed) for the $2\times2$
   case, starting from $A\cdot\text{adj}(A) = \det(A)\,I$. Show this identity holds directly by
   matrix multiplication.

<details>
<summary>💡 Solution outline</summary>

For $A=\begin{pmatrix}a&b\\c&d\end{pmatrix}$, the $2\times2$ adjugate is
$\text{adj}(A)=\begin{pmatrix}d&-b\\-c&a\end{pmatrix}$ (swap the diagonal, negate the
off-diagonal). Multiply directly:
$$A\cdot\text{adj}(A) = \begin{pmatrix}a&b\\c&d\end{pmatrix}\begin{pmatrix}d&-b\\-c&a\end{pmatrix}
= \begin{pmatrix}ad-bc & -ab+ba\\cd-dc & -cb+da\end{pmatrix} =
\begin{pmatrix}ad-bc&0\\0&ad-bc\end{pmatrix} = (ad-bc)I = \det(A)\,I.$$
Both off-diagonal entries collapse to exactly zero by cancellation ($-ab+ba=0$,
$cd-dc=0$), leaving $\det(A)$ on the diagonal. Dividing both sides by the scalar $\det(A)$
(valid whenever $\det(A)\ne0$) gives $A\cdot\frac{\text{adj}(A)}{\det(A)}=I$, i.e.
$A^{-1}=\frac{\text{adj}(A)}{\det(A)}$ — exactly the $2\times2$ inverse formula the Theory
section states, now derived rather than asserted. Numeric check with $A=\begin{pmatrix}4&7\\
2&6\end{pmatrix}$: $A\cdot\text{adj}(A)=\begin{pmatrix}10&0\\0&10\end{pmatrix}=\det(A)\,I$
exactly (since $\det(A)=10$), and $\text{adj}(A)/\det(A)$ matches `np.linalg.inv(A)` exactly.

</details>

### 🐍 IMPLEMENT
4. Extend `det_3x3`'s cofactor-expansion approach to a general recursive `det_cofactor_general(A)`
   that works for any $n\times n$ matrix by expanding along the first row and recursing on
   $(n-1)\times(n-1)$ minors (base case: return the single entry when $n=1$).

<details>
<summary>✅ How to know you're right</summary>

Your function's output must match `np.linalg.det(A)` to floating-point precision for random
matrices of at least 4 different sizes ($n=2,3,4,5$, say) — determinant computation has no
ambiguity or approximation involved, so any mismatch is a bug (a likely culprit: forgetting the
alternating $(-1)^j$ cofactor sign, or an off-by-one in which row/column gets deleted for each
minor). As a structural sanity check independent of comparing to NumPy, verify your own
implementation's output changes sign when you swap any two rows of the input — a determinant's
defining alternating property.

</details>

### 🤖 APPLY
5. Normalizing flows need every layer's Jacobian determinant computed cheaply. Build two
   $n\times n$ lower-triangular matrices $L_1,L_2$ (standing in for two flow layers' Jacobians)
   with nonzero diagonals, and confirm that (a) each one's determinant equals the product of its
   diagonal entries — an $O(n)$ computation instead of general cofactor expansion's much more
   expensive cost — and (b) the *composition* $L_2L_1$ is itself lower-triangular, so its
   determinant is *also* just a diagonal product, consistent with $\det(L_2L_1)=\det(L_2)\det(L_1)$.

<details>
<summary>✅ What you should observe</summary>

For a random $4\times4$ lower-triangular $L_1$ (e.g. diagonal $\approx(0.53,1.03,2.07,0.96)$),
the product of the diagonal ($\approx1.092$) matches `np.linalg.det(L_1)` exactly. $L_2 @ L_1$
has every entry strictly above the diagonal equal to exactly $0.0$ — confirming the product of
two lower-triangular matrices is lower-triangular, a general structural fact, not a coincidence
of this example — and its diagonal's product ($\approx2.067$) matches both
`np.linalg.det(L_2 @ L_1)` directly and $\det(L_2)\times\det(L_1)$ computed separately. This is
exactly why flow architectures (e.g. RealNVP, Glow) deliberately design each layer's Jacobian to
be triangular: composing many layers keeps the *whole* transformation's Jacobian triangular too,
so the determinant needed for the change-of-variables formula in the model's log-likelihood stays
an $O(n)$ diagonal product at every depth, never requiring an expensive general $O(n^3)$
determinant no matter how many layers are stacked.

</details>

### 🚀 CHALLENGE
6. Build a $3\times3$ matrix $A$ that is exactly singular (row 3 = row 1 + row 2), confirm
   `np.linalg.inv` raises an error while `np.linalg.pinv` still returns a result. Then
   investigate: does `np.linalg.pinv` behave *continuously* as a matrix approaches this singular
   $A$ — i.e. does $\text{pinv}(A+\epsilon E)$ converge to $\text{pinv}(A)$ as $\epsilon\to0$ for
   a small random perturbation $E$? Test at several shrinking $\epsilon$ and report what you
   find.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer confirms the pseudoinverse *diverges*, not converges, as the perturbed matrix
approaches the singular one: with $A$'s own pseudoinverse having Frobenius norm
$\approx2.22$, perturbing by $\epsilon=10^{-2}, 10^{-4}, 10^{-6}, 10^{-8}$ in a fixed random
direction produces perturbed pseudoinverses with norms growing to roughly $8.0\times10^2$,
$7.9\times10^4$, $7.9\times10^6$, and $7.9\times10^8$ respectively — growing in inverse
proportion to $\epsilon$, with no sign of leveling off. This is a genuinely counterintuitive
result worth stating explicitly: the pseudoinverse is a *discontinuous* function of the matrix
at exactly the points where rank drops (here, from rank 3 for any $\epsilon\ne0$ perturbation
down to the exactly-singular $A$'s rank 2) — "the pseudoinverse always exists, unlike the
inverse" does not mean "the pseudoinverse is a well-behaved, robust generalization of the
inverse near singular matrices." A strong answer connects this to a real practical consequence:
any computation that relies on `pinv` of a matrix that is only *nearly* rank-deficient (common
with real, noisy data — e.g. a near-collinear design matrix) can be numerically unstable in
exactly the way `01.05`'s own near-singular $A^{-1}$ example already demonstrates for ordinary
inversion, and the fix in both cases is the same idea explored in `01.04`'s regularization
exercise: don't rely on the exact algebraic object near a discontinuity, use a regularized or
truncated version instead.

</details>

---

## 📚 Further Reading
- Strang, *Introduction to Linear Algebra*, Ch. 5 (Determinants)
- Papamakarios et al., ["Normalizing Flows for Probabilistic Modeling and Inference"](https://arxiv.org/abs/1912.02762)

---

*Next notebook: [Eigenvalues and Eigenvectors](06_eigenvalues_eigenvectors.ipynb)*